# บทที่ 7 — อ่านผลอย่างมีวิจารณญาณ

<sub>บทเรียนที่ 7 จาก 8 &nbsp;·&nbsp; [← บทที่ 6](06_comparing_architectures.ipynb) · [สารบัญ](README.md) · [บทที่ 8 →](08_from_hi_to_rul.ipynb)</sub>

## เป้าหมายของบทนี้

เมื่อจบบทนี้คุณจะ:

- รู้ว่า RMSE ที่ดูน้อยอาจไม่ได้แปลว่าโมเดลเก่ง
- สร้าง baseline เพื่อเป็นเกณฑ์เปรียบเทียบ
- เข้าใจว่า R² บอกอะไรที่ RMSE บอกไม่ได้
- ระบุข้อจำกัดของงานตัวเองได้อย่างซื่อสัตย์

> **บทนี้ไม่ต้องใช้ข้อมูลดิบ** — ใช้ไฟล์ `outputs/features_cache.npz` ที่อยู่ใน repo อยู่แล้ว รันได้เลย

---

## 7.1 บทเรียนที่สำคัญที่สุดในซีรีส์นี้

สมมติมีคนบอกคุณว่า *"โมเดลผมได้ RMSE 0.008"*

**คำถามแรกที่ควรถามคือ: แล้วมันดีหรือเปล่า?**

คำตอบคือ**ยังบอกไม่ได้** จนกว่าจะรู้ว่าค่าที่ต้องทายมันกระจายกว้างแค่ไหน

- ถ้าค่าที่ต้องทายอยู่ระหว่าง 0 ถึง 1000 → RMSE 0.008 คือเทพมาก
- ถ้าค่าที่ต้องทายอยู่ระหว่าง 0.39 ถึง 0.41 → RMSE 0.008 คือแทบไม่ต่างจากเดา

In [ ]:
# ── ตั้งค่าให้ notebook มองเห็นโค้ดใน src/ ──
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ── หาฟอนต์ที่แสดงภาษาไทยได้ ไม่งั้นข้อความในกราฟจะกลายเป็นสี่เหลี่ยม ──
_installed = {f.name for f in fm.fontManager.ttflist}
for _f in ["Noto Sans Thai", "Leelawadee UI", "Tahoma", "TH Sarabun New", "Angsana New"]:
    if _f in _installed:
        plt.rcParams["font.family"] = _f
        plt.rcParams["axes.unicode_minus"] = False    # ฟอนต์ไทยมักไม่มีเครื่องหมายลบแบบ unicode
        print("ฟอนต์กราฟ:", _f)
        break
else:
    print("[หมายเหตุ] ไม่พบฟอนต์ไทย - ข้อความไทยในกราฟอาจแสดงเป็นสี่เหลี่ยม")
    print("           Windows/macOS มักมีอยู่แล้ว ส่วน Linux ลง: sudo apt install fonts-thai-tlwg")

print("project root:", ROOT)

In [ ]:
import json
from src.paths import RESULTS_JSON

results = json.load(open(RESULTS_JSON, encoding="utf-8"))
targets = np.array(results["BiLSTM"]["test_targets"])

print("ผลจากการเทรนจริงของโปรเจกต์:")
for name in sorted(results, key=lambda n: results[n]["rmse"]):
    r = results[name]
    print(f"  {name:<13} RMSE={r['rmse']:.4f}  MAE={r['mae']:.4f}  r={r['pearson_r']:.4f}")

## 7.2 ดูการกระจายของคำตอบก่อน

ก่อนตัดสินว่าโมเดลเก่งแค่ไหน ต้องดูก่อนว่าโจทย์ยากแค่ไหน

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(targets, bins=40, color="#1976d2", alpha=0.8)
axes[0].set_xlabel("Health Index")
axes[0].set_ylabel("จำนวนตัวอย่าง")
axes[0].set_title("การกระจายของคำตอบใน test set")

axes[1].plot(np.sort(targets)[::-1], color="#1976d2", linewidth=2)
axes[1].set_xlabel("ตัวอย่าง (เรียงจากมากไปน้อย)")
axes[1].set_ylabel("Health Index")
axes[1].set_title("เรียงลำดับแล้วเห็นชัดว่ากระจุกตัว")

plt.tight_layout()
plt.show()

median = np.median(targets)
for w in (0.01, 0.02, 0.05):
    n = np.sum(np.abs(targets - median) <= w)
    print(f"อยู่ในช่วง +/-{w:.2f} รอบค่ากลาง: {n:>3} / {len(targets)}  ({n/len(targets)*100:.1f}%)")

**นี่คือหัวใจของบทนี้** — คำตอบส่วนใหญ่กระจุกอยู่ในช่วงแคบมาก

เหตุผลคือลูกปืนปกติดีเกือบตลอดการทดลอง Health Index จึงนิ่งอยู่แถว ๆ เดิม
มีแค่ช่วงท้ายที่ค่าตกลงมา

แปลว่า**แค่ทายค่าเดิมทุกครั้งก็ได้คะแนนไม่เลวแล้ว** — มาพิสูจน์กัน

## 7.3 สร้าง Baseline

**Baseline** คือวิธีทำนายที่โง่ที่สุดเท่าที่จะคิดได้ ใช้เป็นเกณฑ์เปรียบเทียบ

ถ้าโมเดลที่เราอุตส่าห์เทรนไม่ชนะ baseline แสดงว่ามันไม่ได้เรียนรู้อะไรเลย

สำหรับปัญหา regression baseline มาตรฐานคือ **ทายค่าเฉลี่ยเสมอ**
(เป็นค่าคงที่ที่ให้ MSE ต่ำที่สุดในทางคณิตศาสตร์)

In [ ]:
baseline_pred = np.full_like(targets, targets.mean())

base_rmse = float(np.sqrt(np.mean((baseline_pred - targets) ** 2)))
base_mae  = float(np.mean(np.abs(baseline_pred - targets)))

print(f"Baseline (ทายค่าเฉลี่ยเสมอ):  RMSE = {base_rmse:.4f}   MAE = {base_mae:.4f}")
print()
print(f"{'Model':<14}{'RMSE':>9}{'ดีกว่า baseline':>18}")
print("=" * 42)
print(f"{'[baseline]':<14}{base_rmse:>9.4f}{'-':>18}")
for name in sorted(results, key=lambda n: results[n]["rmse"]):
    rmse = results[name]["rmse"]
    gain = (base_rmse - rmse) / base_rmse * 100
    print(f"{name:<14}{rmse:>9.4f}{gain:>17.0f}%")

**ตอนนี้ตัวเลขเริ่มมีความหมายแล้ว**

โมเดลชนะ baseline อย่างชัดเจน แปลว่ามัน**เรียนรู้สัญญาณการเสื่อมได้จริง**
ไม่ได้แค่ทายค่ากลาง

แต่สังเกตว่า "ดีกว่า 70%" ให้ความรู้สึกต่างจาก "RMSE 0.0079" มาก
ทั้งที่เป็นตัวเลขชุดเดียวกัน

In [ ]:
plt.figure(figsize=(11, 4))
names = sorted(results, key=lambda n: results[n]["rmse"])
colors = {"CNN-LSTM": "#4f8ef7", "LSTM": "#f97316",
          "BiLSTM": "#a78bfa", "Transformer": "#f472b6"}

plt.bar(["baseline\n(ทายค่าเฉลี่ย)"] + names,
        [base_rmse] + [results[n]["rmse"] for n in names],
        color=["#9e9e9e"] + [colors[n] for n in names])
plt.axhline(base_rmse, color="#616161", linestyle="--", linewidth=1.4)
plt.ylabel("RMSE (ต่ำ = ดี)")
plt.title("ทุกโมเดลต้องอยู่ใต้เส้นประถึงจะมีประโยชน์")
plt.tight_layout()
plt.show()

## 7.4 R² — ตัวเลขที่พกบริบทมาด้วย

ปัญหาของ RMSE คือมัน**ขึ้นกับสเกลของข้อมูล** จึงเอาไปเทียบข้ามงานไม่ได้

**R² (coefficient of determination)** แก้ปัญหานี้ด้วยการเทียบกับ baseline ในตัว:

```
R² = 1 − (ความผิดพลาดของโมเดล / ความผิดพลาดของ baseline)²
```

| ค่า R² | ความหมาย |
|---|---|
| 1.0 | ทำนายถูกเป๊ะทุกตัว |
| 0.0 | พอ ๆ กับทายค่าเฉลี่ย |
| ติดลบ | แย่กว่าทายค่าเฉลี่ย |

In [ ]:
def r2_score(targets, preds):
    ss_res = np.sum((targets - preds) ** 2)
    ss_tot = np.sum((targets - targets.mean()) ** 2)
    return 1 - ss_res / ss_tot


print(f"{'Model':<14}{'RMSE':>9}{'R2':>9}{'ดีกว่า baseline':>18}")
print("=" * 51)
for name in sorted(results, key=lambda n: results[n]["rmse"]):
    preds = np.array(results[name]["test_preds"])
    tg = np.array(results[name]["test_targets"])
    gain = (base_rmse - results[name]["rmse"]) / base_rmse * 100
    print(f"{name:<14}{results[name]['rmse']:>9.4f}{r2_score(tg, preds):>9.4f}{gain:>17.0f}%")

print("\nสังเกตว่า R2 กับ '% ดีกว่า baseline' เล่าเรื่องเดียวกัน")
print("เพราะทั้งคู่เทียบกับ baseline เหมือนกัน แค่คิดคนละสูตร")

> **สิ่งที่ควรรายงานเวลาเขียนรายงานหรือส่งอาจารย์:**
> RMSE เพียว ๆ ไม่พอ ควรรายงาน **R²** หรือ **% ที่ดีกว่า baseline** ควบคู่ไปด้วยเสมอ
> เพราะตัวเลขพวกนี้พกบริบทมาให้ผู้อ่านตัดสินได้เอง

## 7.5 Pearson r หลอกได้

`r` วัดว่าค่าที่ทำนายกับค่าจริง**ขึ้นลงไปด้วยกัน**แค่ไหน
แต่มันไม่สนใจว่าค่าจะตรงกันหรือเปล่า

In [ ]:
tg = np.array(results["BiLSTM"]["test_targets"])

fake_shifted = tg + 0.15          # บวกทุกค่าเท่า ๆ กัน - ผิดหมดแต่ขึ้นลงตาม
fake_scaled  = tg * 3.0           # คูณสามเท่า - ผิดหมดแต่ขึ้นลงตาม

from scipy.stats import pearsonr
print(f"{'กรณี':<28}{'r':>9}{'RMSE':>10}{'R2':>10}")
print("=" * 58)
for name, p in [("ทำนายเลื่อนไป +0.15", fake_shifted),
                ("ทำนายคูณ 3 เท่า", fake_scaled),
                ("BiLSTM ของจริง", np.array(results["BiLSTM"]["test_preds"]))]:
    print(f"{name:<28}{pearsonr(tg, p)[0]:>9.4f}{np.sqrt(np.mean((p-tg)**2)):>10.4f}{r2_score(tg, p):>10.4f}")

สองแถวบนได้ **r = 1.0000 เป๊ะ** ทั้งที่ทำนายผิดหมดทุกค่า!

**บทเรียน:** `r` บอกแค่ทิศทาง ไม่ได้บอกความถูกต้อง
ต้องดูคู่กับ RMSE และ R² เสมอ

## 7.6 ข้อจำกัดของโปรเจกต์นี้

การระบุข้อจำกัดของงานตัวเองไม่ใช่จุดอ่อน แต่เป็นสิ่งที่ทำให้งานน่าเชื่อถือ

**1. Label สร้างขึ้นเอง**
Health Index คำนวณจาก feature ชุดเดียวกับที่ป้อนเข้าโมเดล จึงสัมพันธ์กันอยู่ก่อนแล้ว
→ ตัวเลขห้ามเอาไปเทียบตรง ๆ กับงานที่มีข้อมูลอายุจริง

**2. Label กระจุกตัว**
คำตอบส่วนใหญ่อยู่ในช่วงแคบ → RMSE ดิบดูต่ำเกินจริง ต้องอ่านคู่กับ baseline

**3. หน้าต่างซ้อนทับกัน**
หน้าต่างที่ติดกันใช้ข้อมูลร่วมกัน 19/20 จุด พอสุ่มสลับแล้วแบ่ง
→ Test ไม่เป็นอิสระจาก Train เต็มที่

**4. ทดสอบบนการทดลองเดียว**
ใช้เฉพาะ 2nd_test ยังไม่ได้ยืนยันกับ 1st/3rd_test หรือเครื่องจักรอื่น

**5. รันครั้งเดียวต่อโมเดล**
ยังไม่ได้ทำหลาย seed จึงยังบอกไม่ได้ว่าความต่างที่เห็นเกินความผันผวนจากการสุ่มหรือไม่

In [ ]:
# ข้อ 5 ทดสอบได้ง่าย ๆ - ความต่างระหว่างโมเดลใหญ่แค่ไหนเมื่อเทียบกับ baseline?
rmses = [results[n]["rmse"] for n in results]
spread = max(rmses) - min(rmses)

print(f"ช่วงห่างระหว่างโมเดลดีที่สุดกับแย่ที่สุด: {spread:.4f}")
print(f"เทียบกับ baseline RMSE {base_rmse:.4f} = {spread/base_rmse*100:.0f}% ของ baseline")
print()
print("ถ้าการเทรนซ้ำด้วย seed อื่นให้ผลแกว่งใกล้เคียงตัวเลขนี้")
print("แปลว่าอันดับที่ได้อาจไม่มีความหมายทางสถิติ - ต้องรันหลาย seed ถึงจะสรุปได้")

## 🔧 ลองแก้ดู — ฝึกอ่านผลอย่างมีวิจารณญาณ


1. สร้าง baseline อีกแบบ: **ทายค่าเดิมกับจุดเวลาก่อนหน้า** (persistence baseline)
   ซึ่งมักแข็งแกร่งมากในข้อมูลลำดับเวลา — โมเดลยังชนะไหม?
2. คำนวณ RMSE เฉพาะตัวอย่างที่ Health Index ต่ำกว่าค่ากลาง (ช่วงที่ลูกปืนเริ่มเสื่อม)
   — โมเดลแม่นในช่วงที่*สำคัญจริง*หรือแค่แม่นในช่วงที่ลูกปืนปกติ?
3. ลองคิดว่าถ้าต้องนำเสนอผลนี้ให้วิศวกรโรงงาน คุณจะรายงานตัวเลขไหน เพราะอะไร

## ❓ เช็คความเข้าใจ

**1. ทำไม RMSE เพียว ๆ ถึงเอาไปเทียบข้ามงานวิจัยไม่ได้?**

<details>
<summary>ดูเฉลย</summary>

เพราะ RMSE อยู่ในหน่วยเดียวกับ label ถ้าสองงานใช้ label คนละสเกลหรือคนละนิยาม ตัวเลขก็เทียบกันไม่ได้ ควรใช้ R2 หรือ % ที่ดีกว่า baseline ซึ่งไม่มีหน่วย

</details>

**2. โมเดลที่ได้ R² = 0.05 แปลว่าอะไร ควรเอาไปใช้งานไหม?**

<details>
<summary>ดูเฉลย</summary>

แปลว่าดีกว่าการทายค่าเฉลี่ยแค่ 5% ในเชิงความแปรปรวน — แทบไม่ได้เรียนรู้อะไรเลย ไม่ควรนำไปใช้ และควรกลับไปทบทวนว่า feature เพียงพอไหม หรือปัญหานี้ทำนายได้จริงหรือเปล่า

</details>

**3. ถ้า Pearson r = 0.99 แต่ R² = -2.0 เป็นไปได้ไหม เกิดจากอะไร?**

<details>
<summary>ดูเฉลย</summary>

เป็นไปได้ เกิดเมื่อค่าที่ทำนายขึ้นลงตามค่าจริงเป๊ะ (r สูง) แต่มี offset หรือ scale ผิด เช่นทำนายสูงกว่าความจริงตลอด R2 จึงติดลบเพราะความผิดพลาดมากกว่าการทายค่าเฉลี่ยเสียอีก

</details>

---

## สรุปบทนี้

- ตัวเลขความผิดพลาดไม่มีความหมายจนกว่าจะเทียบกับ baseline
- baseline ที่ง่ายที่สุดคือทายค่าเฉลี่ยเสมอ — โมเดลต้องชนะให้ได้
- รายงาน R² หรือ % ที่ดีกว่า baseline คู่กับ RMSE เสมอ
- Pearson r บอกแค่ทิศทาง ค่า 1.0 ไม่ได้แปลว่าทำนายถูก
- ระบุข้อจำกัดของงานตัวเองให้ชัด — มันทำให้งานน่าเชื่อถือขึ้น ไม่ใช่ลดลง

[← บทที่ 6](06_comparing_architectures.ipynb) &nbsp;·&nbsp; [สารบัญ](README.md) &nbsp;·&nbsp; **[บทที่ 8 — จาก HI สู่ RUL →](08_from_hi_to_rul.ipynb)**